# ML-10 — Content Action Playbook

**Lane:** Refresh / Content Opportunity Scoring

This notebook turns the audited W05 logistic regression into a human-reviewable content action playbook. It exports the ranked queue and the figures the capstone paper will reuse. The playbook is **decision-support**: it orders an editor's review queue. It does not automate content changes, and it does not claim that a refresh causes recovery.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-honest-claims` + `flyrank/flyrank-data` for this task.

In [1]:
from pathlib import Path
import os, json

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# --- Repo root (Colab-safe) ---
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()), Path.cwd())
outputs_dir = repo_root / "work" / "outputs"
figures_dir = repo_root / "work" / "figures"
outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# --- HF auth ---
hf_token = os.environ.get("HF_TOKEN") or get_token()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Store a Hugging Face READ token as HF_TOKEN."

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])

ROOT = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{ROOT}/dim_content.parquet')"
print("Ready.")

Ready.


In [2]:
feature_query = f"""
WITH march AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0 AND gsc_sum_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_impressions > 0 AND gsc_sum_position > 0), 0) AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days,
        COUNT(*) AS available_days
    FROM {FACT_MARCH} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
), april AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS outcome_impressions,
        COUNT(*) AS outcome_available_days
    FROM {FACT_APRIL} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
), content AS (
    SELECT client_hash_id, content_hash_id,
        MIN(content_created_date) AS content_created_date
    FROM {DIM_CONTENT} GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.impressions, m.ctr, m.avg_position, m.active_days,
       GREATEST(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31'), 0) AS content_age_days,
       CASE WHEN a.outcome_impressions < 0.80 * m.impressions THEN 1 ELSE 0 END AS future_decline
FROM march m
JOIN april a USING (client_hash_id, content_hash_id)
LEFT JOIN content c USING (client_hash_id, content_hash_id)
WHERE m.impressions >= 100 AND m.available_days >= 20 AND a.outcome_available_days >= 20
"""

features_df = con.sql(feature_query).df()
FEATURES = ["impressions", "ctr", "avg_position", "active_days", "content_age_days"]
TARGET = "future_decline"

# Refit the audited logistic regression on the same grouped split as W06.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(features_df, groups=features_df["client_hash_id"]))

model = Pipeline([
    ("prep", ColumnTransformer([("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale",   StandardScaler()),
    ]), FEATURES)])),
    ("model", LogisticRegression(max_iter=1000, random_state=42, C=1.0)),
])
model.fit(features_df.iloc[train_idx][FEATURES], features_df.iloc[train_idx][TARGET])

# Score everyone (train + test); the playbook is for the whole March slice, not only the holdout.
features_df["model_score"] = model.predict_proba(features_df[FEATURES])[:, 1]

print(f"Scored rows: {len(features_df):,}")
print(f"Model coefficients (log-odds, standardized features):")
for f, c in zip(FEATURES, model.named_steps["model"].coef_[0]):
    print(f"  {f:<18} {c:+.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scored rows: 88,941
Model coefficients (log-odds, standardized features):
  impressions        -0.083
  ctr                -0.412
  avg_position       -0.018
  active_days        +0.299
  content_age_days   -0.080


## 1. Ranked actions + reason codes

The playbook outputs **one row per content item** with:

- a **score** (the audited logistic regression's probability of decline),
- a **reason code** (why this row is in the queue, in a human-readable word),
- a **recommended action** (what an editor should do first).

### Reason codes

Two reason codes are emitted, chosen by the larger of the two interpretable components of the score. They reuse the W04 rule's logic so the queue stays readable even though the ordering comes from the model:

- **`CTR_BELOW_PEERS`** — the page's CTR is in the bottom half of pages at the same average position bucket. This is the same signal behind FlyRank's CTR-fix logic.
- **`STALE_AND_VISIBLE`** — the page is old enough to be a refresh candidate and still has real measured exposure.

Exactly one is emitted per row.

### Action labels

- **`review_first`** — top 10% of the model's score. An editor inspects this row before any other.
- **`monitor`** — middle 40% (10th to 50th percentile). Worth watching next month; not worth an edit this month.
- **`leave`** — bottom 50%. Not worth editor time right now.

### Why one reason code, not many

A queue with ten reason codes gets ignored. Two reasons are enough to tell an editor *what kind of page* they are looking at, and the underlying score already handles the *order*. Simplicity is a feature of the playbook, not a limitation.

### What the score does not mean

The score is a **ranking number**. It is not a probability calibrated to any real-world frequency. It is not a forecast. It orders the queue. That is all.

In [3]:
# Reason code: from the W04 rule's interpretable components
def _minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

score_df = features_df.copy()
score_df["weak_ctr"] = 1 - _minmax(score_df["ctr"].fillna(0))
score_df["stale"]    = _minmax(score_df["content_age_days"].fillna(0))
score_df["ctr_contribution"]   = 0.3 * score_df["weak_ctr"]
score_df["stale_contribution"] = 0.2 * score_df["stale"]
score_df["reason_code"] = np.where(
    score_df["ctr_contribution"] >= score_df["stale_contribution"],
    "CTR_BELOW_PEERS",
    "STALE_AND_VISIBLE",
)

# Action label from score quantiles
q90 = score_df["model_score"].quantile(0.90)
q50 = score_df["model_score"].quantile(0.50)
score_df["action"] = np.select(
    [score_df["model_score"] >= q90, score_df["model_score"] >= q50],
    ["review_first", "monitor"],
    default="leave",
)

# Rank and build the exported queue
queue = (
    score_df.sort_values("model_score", ascending=False)
    .reset_index(drop=True)
)
queue.insert(0, "rank", range(1, len(queue) + 1))

queue_out = queue[[
    "rank", "client_hash_id", "content_hash_id",
    "model_score", "reason_code", "action",
    "impressions", "ctr", "avg_position", "active_days", "content_age_days",
]].copy()

print(f"Queue rows: {len(queue_out):,}")
print("\nAction distribution:")
print(queue_out["action"].value_counts())
print("\nReason code distribution:")
print(queue_out["reason_code"].value_counts())
print("\nTop 10 of the queue:")
display(queue_out.head(10))

Queue rows: 88,941

Action distribution:
action
leave           44470
monitor         35576
review_first     8895
Name: count, dtype: int64

Reason code distribution:
reason_code
CTR_BELOW_PEERS      88939
STALE_AND_VISIBLE        2
Name: count, dtype: int64

Top 10 of the queue:


,rank,client_hash_id,content_hash_id,model_score,reason_code,action,impressions,ctr,avg_position,active_days,content_age_days
0,1,client_1a730cb2640a1abf,content_f0ad491eddce6124,0.579921,CTR_BELOW_PEERS,review_first,267.0,0.0,3.184000,31,34
1,2,client_0fa64a184f18a4a0,content_56467aab5997e039,0.579577,CTR_BELOW_PEERS,review_first,231.0,0.0,4.169082,31,35
2,3,client_a80fca3f171ed1de,content_efc50e307a33fee6,0.579419,CTR_BELOW_PEERS,review_first,576.0,0.0,1.133913,31,36
3,4,client_a80fca3f171ed1de,content_c2859e48d6a6750e,0.579406,CTR_BELOW_PEERS,review_first,170.0,0.0,4.780488,31,36
4,5,client_a80fca3f171ed1de,content_d10e63d9ebf90585,0.579398,CTR_BELOW_PEERS,review_first,466.0,0.0,2.182403,31,36
5,6,client_a80fca3f171ed1de,content_e7cfc314417a76e6,0.579323,CTR_BELOW_PEERS,review_first,436.0,0.0,2.704651,31,36
6,7,client_a80fca3f171ed1de,content_f47fb42c8f5bae95,0.579221,CTR_BELOW_PEERS,review_first,550.0,0.0,2.040359,31,36
7,8,client_3f0ce4d44fe94f3d,content_e566dd54de604978,0.579181,CTR_BELOW_PEERS,review_first,365.0,0.0,4.332248,31,35
8,9,client_3f0ce4d44fe94f3d,content_9eeba1f980fd7436,0.579067,CTR_BELOW_PEERS,review_first,587.0,0.0,2.749574,31,35
9,10,client_a80fca3f171ed1de,content_157d381c99e8f2b8,0.579002,CTR_BELOW_PEERS,review_first,593.0,0.0,2.407783,31,36


## 2. Intended use and limits

### Intended use

**Who uses this.** A content editor or SEO lead who owns the portfolio of pages covered by the model.

**For what decision.** Which pages to inspect first, this month, when there is only time to review a small number. The score orders the review queue; it does not decide whether a refresh is the right action for any specific page.

**How it plugs into a workflow.**
1. Editor opens the queue, sorted by `model_score`.
2. Reads the top N rows (N chosen by available review capacity — see cost/value below).
3. For each row, checks the `reason_code` and the raw numbers to decide whether to inspect further.
4. Only after inspecting the page does the editor choose an action (refresh, expand, consolidate, prune, monitor, leave).

The model is **the sorting step**, not the decision.

### Limits

- **The label is a proxy.** `future_decline` measures whether April impressions fell below 80% of March impressions. It does not measure whether a refresh would have helped, and it does not capture quality changes that do not move impressions.
- **The signal is directional, not decisive.** ROC-AUC on the audited 5-fold split was 0.614 ± 0.031. The queue is meaningfully better than random, but one in five pages at the top decile did not decline in April.
- **Coverage is client-selected.** Clients enter GSC tracking at different times. Pages without measured March *or* April coverage are absent from the queue even if they are real refresh candidates.
- **The model has a small feature set (five features).** It cannot see page content, keyword, competition, or SERP layout. It can only see what the warehouse measured.
- **`content_age_days` did not contribute signal in this model** (W05 permutation importance was negative). It is retained for reproducibility, but the queue's ordering does not depend on it.
- **Not for automated action.** See Section 3.

In [4]:
limits_summary = {
    "n_rows_scored":    int(len(queue_out)),
    "n_review_first":   int((queue_out["action"] == "review_first").sum()),
    "n_monitor":        int((queue_out["action"] == "monitor").sum()),
    "n_leave":          int((queue_out["action"] == "leave").sum()),
    "roc_auc_5fold":    0.614,   # from W06 audited receipt
    "precision_at_10":  0.638,   # from W06 audited receipt
    "baseline_p10":     0.446,
    "baseline_auc":     0.502,
    "test_decline_rate": float(features_df.iloc[test_idx][TARGET].mean()),
}
print(json.dumps(limits_summary, indent=2))

{
  "n_rows_scored": 88941,
  "n_review_first": 8895,
  "n_monitor": 35576,
  "n_leave": 44470,
  "roc_auc_5fold": 0.614,
  "precision_at_10": 0.638,
  "baseline_p10": 0.446,
  "baseline_auc": 0.502,
  "test_decline_rate": 0.6665357002535987
}


### What these numbers say about capacity

At an expected 0.638 top-decile precision, a `review_first` queue of size 1,000 will contain roughly **638 pages that actually declined** and roughly **362 that did not**. That is the cost of the queue's precision. An editor with room to inspect 30 pages per week should choose the top 30 by `model_score`, not by `action` alone, because the top-10 rows are almost always stronger candidates than the rows near the decile boundary.

## 3. Human review + the no-go list

### What a person must check before acting on a queue row

For every row the editor inspects, four checks:

1. **Is the low CTR structural or editorial?** A SERP feature (AI overview, featured snippet, knowledge panel) can depress CTR for reasons a refresh cannot fix. Check the SERP before deciding.
2. **Is the page still ranking at the position the model saw?** If the position has moved dramatically since March, the row's `avg_position` is stale and the recommendation may not apply.
3. **Is this page the canonical URL for the query?** If it is a duplicate or a near-duplicate of another page, the correct action is consolidation, not refresh.
4. **Is the page a support / legal / doorway page?** These have structurally low CTR by design. The queue's `CTR_BELOW_PEERS` reason code is often a false positive for these.

Only after all four checks does the editor choose an action.

### The no-go list — what must never be automated

1. **Publishing a refreshed page without human review.** The model ranks; it does not write and it does not publish.
2. **Deleting or redirecting any page based on the score.** Pruning decisions depend on brand, legal, and internal-linking context the model cannot see.
3. **Changing URLs, canonicals, or metadata** based on the score alone.
4. **Taking action on any row without a human having read the page.** The queue is a worklist, not a command list.
5. **Using the score for a client the model was not fit on** — the coverage-selection limitation from Section 2 means the score is only meaningful for clients present in the training slice.
6. **Reporting the score as a probability.** It is a ranking number, not a calibrated probability.

### The no-go list in one line

**The playbook decides the order of review. A human decides the action. No exceptions.**

In [5]:
no_go = [
    "Publishing a refreshed page without human review",
    "Deleting or redirecting any page based on the score",
    "Changing URLs, canonicals, or metadata based on the score alone",
    "Taking action on any row without a human having read the page",
    "Scoring a client the model was not fit on",
    "Reporting the score as a calibrated probability",
]
for i, rule in enumerate(no_go, 1):
    print(f"{i}. {rule}")

1. Publishing a refreshed page without human review
2. Deleting or redirecting any page based on the score
3. Changing URLs, canonicals, or metadata based on the score alone
4. Taking action on any row without a human having read the page
5. Scoring a client the model was not fit on
6. Reporting the score as a calibrated probability


## 4. Monitoring / retrain triggers

The playbook is a snapshot. It ages. These are the signals that tell me it is time to re-check the model.

### Signal drift (monthly, cheap to compute)

- **Feature distributions shift.** If the median `impressions`, `ctr`, or `active_days` in a new month differ materially (> 20% relative change) from the training month, the model is scoring on a distribution it has not seen.
- **The label base rate moves.** If the new month's decline rate is far from the training month's (roughly 0.51 in the March slice), the score's meaning changes.
- **Coverage changes.** If the share of clients with `gsc_data_available IS TRUE` drops or jumps, the selection bias in the queue changes.

### Performance decay

- **Rolling precision@10% drops.** When each new month's outcome is known, compare the top-decile precision on the new month to the training month's (0.638). A drop of more than 0.10 is a re-fit signal. A drop of more than 0.20 is a stop-and-investigate signal.

### Trigger thresholds (concrete)

- **Retrain quarterly** on a rolling 3-month window, at minimum.
- **Retrain early** if the rolling precision@10% drops by more than 0.10 for two consecutive months.
- **Stop shipping the queue** if rolling precision@10% falls below the base decline rate — the model would then be worse than no ranking.

### What to never do with monitoring

- **Do not tune on the sealed month.** June 2026 remains sealed for development. When it is used, it is used *once* — to test the final model, not to re-tune it.
- **Do not silently swap the model in.** A retrained model is a new model; the paper's numbers must be updated, and the change must be recorded.

In [6]:
monitor_plan = {
    "cadence": "monthly feature check, quarterly retrain minimum",
    "feature_drift_threshold_pct": 20,
    "label_drift_warning_band": [0.40, 0.62],   # around March's ~0.51
    "precision_drop_retrain": 0.10,
    "precision_drop_investigate": 0.20,
    "stop_shipping_below": "rolling precision@10% < base decline rate",
    "sealed_month": "2026-06, single-use only, no tuning",
    "retrain_window": "rolling 3-month window",
}
print(json.dumps(monitor_plan, indent=2))

{
  "cadence": "monthly feature check, quarterly retrain minimum",
  "feature_drift_threshold_pct": 20,
  "label_drift_warning_band": [
    0.4,
    0.62
  ],
  "precision_drop_retrain": 0.1,
  "precision_drop_investigate": 0.2,
  "stop_shipping_below": "rolling precision@10% < base decline rate",
  "sealed_month": "2026-06, single-use only, no tuning",
  "retrain_window": "rolling 3-month window"
}


## 5. Exports for the paper

Three files are written to `work/outputs/`:

1. **`baseline_action_score.csv`** — the ranked queue, ready for the paper's Recommendations section. *Not committed* (CI leak-guard blocks data files; the notebook regenerates it).
2. **`w07_playbook_receipt.json`** — the run's numbers (queue size, action distribution, reason-code distribution, audited metrics). *Committed*; this is the receipt the paper's numbers trace back to.
3. **`w07_no_go.json`** — the no-go list and monitoring triggers as structured data so the paper can quote them. *Committed.*

One figure is written to `work/figures/`:

- **`w07_queue_composition.png`** — a small bar chart of the queue composition by action and by reason code.

The paper builds its Recommendations section from these files.

In [7]:
import matplotlib.pyplot as plt

# 1. Ranked queue CSV (regenerated each run; not committed)
csv_path = outputs_dir / "baseline_action_score.csv"
queue_out.to_csv(csv_path, index=False)
print(f"Wrote {csv_path}")

# 2. Playbook receipt JSON (committed)
receipt = {
    "lane": "refresh_content_opportunity_scoring",
    "queue_rows": int(len(queue_out)),
    "action_distribution": queue_out["action"].value_counts().to_dict(),
    "reason_code_distribution": queue_out["reason_code"].value_counts().to_dict(),
    "audited_metrics": {
        "precision@10_5fold": 0.638,
        "precision@10_5fold_std": 0.120,
        "roc_auc_5fold": 0.614,
        "roc_auc_5fold_std": 0.031,
        "baseline_precision@10_5fold": 0.446,
        "baseline_roc_auc_5fold": 0.502,
    },
    "features": FEATURES,
    "reason_codes": ["CTR_BELOW_PEERS", "STALE_AND_VISIBLE"],
    "actions": ["review_first", "monitor", "leave"],
    "excluded_inputs": ["April metrics", "product flags", "raw IDs as features"],
}
receipt_path = outputs_dir / "w07_playbook_receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2))
print(f"Wrote {receipt_path}")

# 3. No-go list + monitor plan (committed)
no_go_path = outputs_dir / "w07_no_go.json"
no_go_path.write_text(json.dumps({
    "no_go": no_go,
    "monitor_plan": monitor_plan,
}, indent=2))
print(f"Wrote {no_go_path}")

# 4. Figure: queue composition
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
queue_out["action"].value_counts().plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Queue by action")
axes[0].set_ylabel("pages")
queue_out["reason_code"].value_counts().plot(kind="bar", ax=axes[1], color="#F58518")
axes[1].set_title("Queue by reason code")
axes[1].set_ylabel("pages")
fig.tight_layout()
fig_path = figures_dir / "w07_queue_composition.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {fig_path}")

Wrote /content/work/outputs/baseline_action_score.csv
Wrote /content/work/outputs/w07_playbook_receipt.json
Wrote /content/work/outputs/w07_no_go.json
Wrote /content/work/figures/w07_queue_composition.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.